# Deep Dive: CUDA — Euclidean Norm (C vs CUDA Unified Memory variants)

**Group 3:** Matthew Fabregas, Kane Ibanez, Bowen Li, John Teoxon

**Specification (from our SIMD project):** compute the Euclidean norm of a double-precision vector `A` of length `n`

$$Z = \sqrt{\sum_{i=0}^{n-1} A_i^2}$$

**Data initialization:** `A[i] = sin(i * 0.001)` (double precision), **n = 2^24 = 16,777,216 elements** (128 MB).

**Versions (one cell each):**
1. C program (sequential baseline + correctness answer key)
2. CUDA — grid-stride loop, Unified Memory, no prefetch, no page creation, no mem advise
3. CUDA — grid-stride loop + prefetch (no page creation, no mem advise)
4. CUDA — grid-stride loop + prefetch + page creation (no mem advise)
5. CUDA — grid-stride loop + prefetch + page creation + mem advise

Each kernel is executed **30 times** for block sizes **256, 512 and 1024**. Kernel time is measured with CUDA events (kernel only). Every CUDA result is checked against the C answer key.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

In [1]:
!nvidia-smi
!nvcc --version
!rm -f results.csv

Mon Nov  9 09:01:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Version 1 — C program (baseline and sanity-check answer key)

Same algorithm as `norm_c` in our SIMD project: one sequential loop that adds `A[i] * A[i]`, then `sqrt`. The vector is initialized with `A[i] = sin(i * 0.001)`. One untimed warm-up call, then **30 timed calls** (only the function call is timed, using `clock_gettime`). The resulting `Z` is the answer key for all CUDA versions.

In [2]:
%%writefile euclid_c.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define RUNS 30

double norm_c(const double* A, long long n) {
    double sum = 0.0;
    for (long long i = 0; i < n; i++)
        sum += A[i] * A[i];
    return sqrt(sum);
}

static double now_ms(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1000.0 + t.tv_nsec / 1e6;
}

int main(int argc, char** argv) {
    int p = (argc > 1) ? atoi(argv[1]) : 24;
    long long n = 1LL << p;
    double* A = (double*)malloc(n * sizeof(double));
    for (long long i = 0; i < n; i++)
        A[i] = sin(i * 0.001);

    double z = norm_c(A, n); // warm-up (untimed)
    double total = 0, mn = 1e300, mx = 0, first = 0;
    for (int r = 0; r < RUNS; r++) {
        double t1 = now_ms();
        z = norm_c(A, n);
        double ms = now_ms() - t1;
        if (r == 0) first = ms;
        total += ms;
        if (ms < mn) mn = ms;
        if (ms > mx) mx = ms;
    }
    double avg = total / RUNS;

    printf("VERSION 1: C program (baseline / answer key)\n");
    printf("n = 2^%d = %lld elements, A[i] = sin(i*0.001)\n", p, n);
    printf("Z = %.10f\n", z);
    printf("Execution time over %d runs: avg = %.4f ms | min = %.4f ms | max = %.4f ms\n", RUNS, avg, mn, mx);

    FILE* f = fopen("results.csv", "a");
    if (f) {
        fprintf(f, "C,-,%d,%.6f,%.6f,%.6f,%.6f,%.12f,%.12f,PASS\n", p, first, avg, mn, mx, z, z);
        fclose(f);
    }
    free(A);
    return 0;
}

Writing euclid_c.c


In [3]:
!gcc -O2 -o euclid_c euclid_c.c -lm
!./euclid_c 24

VERSION 1: C program (baseline / answer key)
n = 2^24 = 16777216 elements, A[i] = sin(i*0.001)
Z = 2896.2749920235
Execution time over 30 runs: avg = 43.2398 ms | min = 24.6698 ms | max = 73.2367 ms


## Shared CUDA code (kernel + helpers used by all four CUDA versions)

All four CUDA versions run the **same kernel**, so the only difference between them is how the Unified Memory is handled.

**Kernel (`norm_kernel`)** — a reduction, because every element adds into one sum:
1. **Grid-stride loop:** each thread squares and adds its elements `index, index + stride, ...` into a private register `local`.
2. **Shared memory:** each block adds its threads' `local` values together with a tree reduction (`__syncthreads()` between steps).
3. **atomicAdd:** thread 0 of each block adds the block's total into the global `sum`. Without the atomic, blocks would overwrite each other (race condition, see lecture 4). Only one atomic per block is needed instead of one per element.

The host takes `sqrt(sum)` after the kernel.

**Correctness check:** the reference `Z` is computed with the same C function as Version 1. Because the GPU adds the numbers in a different order, results can differ in the last digits, so a result is `CORRECT` if its relative difference from C is at most `1e-9` (same tolerance as our SIMD project).

**Timing:** for each of the 30 runs, `sum` is reset with `cudaMemset` (not timed), then only the kernel is timed with CUDA events. We report run 1 separately because it includes any page migration.

**Grid size:** `numBlocks = (n + blockSize - 1) / blockSize` (enough blocks for one element per thread).

**CUDA 12 vs 13:** CUDA 13 changed `cudaMemPrefetchAsync` and `cudaMemAdvise` to take a `cudaMemLocation`. The small wrappers below pick the correct call for whatever toolkit Colab has.

In [4]:
%%writefile common.cuh
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define RUNS 30

#define CUDA_CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
    printf("CUDA error %s at line %d: %s\n", #x, __LINE__, cudaGetErrorString(e)); exit(1); } } while (0)

// ---------------- kernels ----------------

// Sum of squares with a grid-stride loop, shared-memory block reduction, one atomicAdd per block.
__global__ void norm_kernel(const double* A, long long n, double* sum) {
    __shared__ double s[1024];
    long long index  = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;

    double local = 0.0;
    for (long long i = index; i < n; i += stride)
        local += A[i] * A[i];

    s[threadIdx.x] = local;
    __syncthreads();
    for (int k = blockDim.x / 2; k > 0; k >>= 1) {   // blockDim is 256/512/1024 (power of 2)
        if (threadIdx.x < k)
            s[threadIdx.x] += s[threadIdx.x + k];
        __syncthreads();
    }
    if (threadIdx.x == 0)
        atomicAdd(sum, s[0]);
}

// Initializes A on the GPU (used by the "page creation" versions), grid-stride loop.
__global__ void init_kernel(double* A, long long n) {
    long long index  = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = index; i < n; i += stride)
        A[i] = sin(i * 0.001);
}

// ---------------- host helpers ----------------

// Same initialization as Version 1 (CPU)
inline void init_host(double* A, long long n) {
    for (long long i = 0; i < n; i++)
        A[i] = sin(i * 0.001);
}

// C answer key: same function as Version 1, computed on a separate host array (not timed)
inline double reference_norm(long long n) {
    double* h = (double*)malloc(n * sizeof(double));
    init_host(h, n);
    double sum = 0.0;
    for (long long i = 0; i < n; i++)
        sum += h[i] * h[i];
    free(h);
    return sqrt(sum);
}

inline void prefetch_to_gpu(const void* p, size_t bytes, int dev) {
#if CUDART_VERSION >= 13000
    cudaMemLocation loc = {};
    loc.type = cudaMemLocationTypeDevice;
    loc.id = dev;
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, loc, 0, 0));
#else
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, dev, 0));
#endif
}

inline void advise_gpu(const void* p, size_t bytes, cudaMemoryAdvise advice, int dev) {
#if CUDART_VERSION >= 13000
    cudaMemLocation loc = {};
    loc.type = cudaMemLocationTypeDevice;
    loc.id = dev;
    CUDA_CHECK(cudaMemAdvise(p, bytes, advice, loc));
#else
    CUDA_CHECK(cudaMemAdvise(p, bytes, advice, dev));
#endif
}

// Runs the kernel RUNS times, times each run, checks correctness, prints and appends to results.csv
inline void run_and_report(const char* name, const char* label, const double* A, long long n,
                           int p, int blockSize, double* sum) {
    int numBlocks = (int)((n + blockSize - 1) / blockSize);
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    double times[RUNS];
    for (int r = 0; r < RUNS; r++) {
        CUDA_CHECK(cudaMemset(sum, 0, sizeof(double)));   // reset sum (not timed)
        CUDA_CHECK(cudaEventRecord(start));
        norm_kernel<<<numBlocks, blockSize>>>(A, n, sum);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaGetLastError());
        float ms = 0;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        times[r] = ms;
    }
    CUDA_CHECK(cudaDeviceSynchronize());
    double z = sqrt(*sum);                                   // result of the last run

    double total = 0, mn = 1e300, mx = 0, warm = 0;
    for (int r = 0; r < RUNS; r++) {
        total += times[r];
        if (times[r] < mn) mn = times[r];
        if (times[r] > mx) mx = times[r];
        if (r > 0) warm += times[r];
    }
    double avg = total / RUNS;
    warm /= (RUNS - 1);

    double ref = reference_norm(n);
    double rel = fabs(z - ref) / fabs(ref);
    bool ok = rel <= 1e-9;

    printf("%s\n", name);
    printf("n = 2^%d = %lld | block size = %d | blocks = %d\n", p, n, blockSize, numBlocks);
    printf("Kernel time over %d runs: avg = %.4f ms | min = %.4f ms | max = %.4f ms\n", RUNS, avg, mn, mx);
    printf("Run 1 = %.4f ms | avg of runs 2-%d = %.4f ms\n", times[0], RUNS, warm);
    printf("Z (CUDA) = %.10f | Z (C answer key) = %.10f | rel diff = %.2e -> %s\n\n",
           z, ref, rel, ok ? "CORRECT" : "WRONG");

    FILE* f = fopen("results.csv", "a");
    if (f) {
        fprintf(f, "%s,%d,%d,%.6f,%.6f,%.6f,%.6f,%.12f,%.12f,%s\n",
                label, blockSize, p, times[0], avg, mn, mx, z, ref, ok ? "PASS" : "FAIL");
        fclose(f);
    }
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
}

// args: blockSize [power p, default 24]
inline void parse_args(int argc, char** argv, int* blockSize, int* p) {
    *blockSize = (argc > 1) ? atoi(argv[1]) : 256;
    *p = (argc > 2) ? atoi(argv[2]) : 24;
}

Writing common.cuh


## Version 2 — CUDA Unified Memory (grid-stride loop, no prefetch, no page creation, no mem advise)

`A` is allocated with `cudaMallocManaged` and initialized on the **CPU**, so its pages are created in CPU memory. Nothing else is done: when the kernel first touches `A`, the GPU page-faults and the pages migrate to the GPU on demand. This migration cost lands inside **run 1** of the kernel.

In [5]:
%%writefile cuda_unified.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, p;
    parse_args(argc, argv, &blockSize, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    init_host(A, n);   // pages created on the CPU; no prefetch, no advise

    run_and_report("VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)",
                   "CUDA Unified", A, n, p, blockSize, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_unified.cu


In [6]:
!nvcc -O2 -arch=sm_75 -o cuda_unified cuda_unified.cu
!for b in 256 512 1024; do ./cuda_unified $b 24; done

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 3.4607 ms | min = 2.2836 ms | max = 37.4334 ms
Run 1 = 37.4334 ms | avg of runs 2-30 = 2.2892 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 512 | blocks = 32768
Kernel time over 30 runs: avg = 3.2900 ms | min = 2.1829 ms | max = 32.7905 ms
Run 1 = 32.7905 ms | avg of runs 2-30 = 2.2728 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.91e-14 -> CORRECT

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 1024 | blocks = 16384
Kernel time over 30 runs: avg = 3.9207 ms | min = 2.8385 ms | max = 35.2356 ms
Run 1 = 35.2356 ms | avg of runs 2-30 = 2.8409 ms
Z (CUDA) = 2896.2749920236 | Z (C a

## Version 3 — CUDA Unified Memory + prefetch (no page creation, no mem advise)

Same as Version 2 (`A` initialized on the **CPU**), but before the kernel we call `cudaMemPrefetchAsync(A, bytes, device)`. The whole vector is copied to the GPU in large chunks **before** the kernel starts, so the kernel itself does not page-fault.

In [7]:
%%writefile cuda_prefetch.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, p;
    parse_args(argc, argv, &blockSize, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    init_host(A, n);                        // pages created on the CPU
    prefetch_to_gpu(A, bytes, device);      // migrate A to the GPU before the kernel
    prefetch_to_gpu(sum, sizeof(double), device);
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 3: CUDA Prefetch (no page creation, no mem advise)",
                   "CUDA Prefetch", A, n, p, blockSize, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_prefetch.cu


In [8]:
!nvcc -O2 -arch=sm_75 -o cuda_prefetch cuda_prefetch.cu
!for b in 256 512 1024; do ./cuda_prefetch $b 24; done

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 2.2967 ms | min = 2.2838 ms | max = 2.6215 ms
Run 1 = 2.6215 ms | avg of runs 2-30 = 2.2855 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 512 | blocks = 32768
Kernel time over 30 runs: avg = 2.8120 ms | min = 2.7873 ms | max = 3.1533 ms
Run 1 = 3.1533 ms | avg of runs 2-30 = 2.8003 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.91e-14 -> CORRECT

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 1024 | blocks = 16384
Kernel time over 30 runs: avg = 3.0398 ms | min = 3.0250 ms | max = 3.3524 ms
Run 1 = 3.3524 ms | avg of runs 2-30 = 3.0291 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 

## Version 4 — CUDA Unified Memory + prefetch + page creation (no mem advise)

**Our definition of page creation:** `cudaMallocManaged` only reserves addresses; the physical pages are created when the memory is first touched. Here we create the pages **directly in GPU memory**: we prefetch the still-empty `A` to the GPU, then initialize it with a GPU kernel (`init_kernel`, same formula `sin(i * 0.001)`). The data is never written by the CPU, so nothing has to be migrated from CPU to GPU at all.

In [9]:
%%writefile cuda_pages.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, p;
    parse_args(argc, argv, &blockSize, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    prefetch_to_gpu(A, bytes, device);      // prefetch the empty buffer to the GPU
    prefetch_to_gpu(sum, sizeof(double), device);
    init_kernel<<<(int)((n + blockSize - 1) / blockSize), blockSize>>>(A, n);  // page creation on the GPU
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 4: CUDA Prefetch + page creation (no mem advise)",
                   "CUDA Prefetch+Page creation", A, n, p, blockSize, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_pages.cu


In [10]:
!nvcc -O2 -arch=sm_75 -o cuda_pages cuda_pages.cu
!for b in 256 512 1024; do ./cuda_pages $b 24; done

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 2.4697 ms | min = 2.4639 ms | max = 2.5518 ms
Run 1 = 2.5518 ms | avg of runs 2-30 = 2.4669 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 512 | blocks = 32768
Kernel time over 30 runs: avg = 2.7943 ms | min = 2.5963 ms | max = 2.8260 ms
Run 1 = 2.8260 ms | avg of runs 2-30 = 2.7932 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.91e-14 -> CORRECT

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 1024 | blocks = 16384
Kernel time over 30 runs: avg = 3.0410 ms | min = 3.0356 ms | max = 3.0604 ms
Run 1 = 3.0604 ms | avg of runs 2-30 = 3.0403 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.89e-

## Version 5 — CUDA Unified Memory + prefetch + page creation + mem advise

Same as Version 4, plus `cudaMemAdvise` hints (lecture 2):
- `cudaMemAdviseSetPreferredLocation` = GPU for `A` and `sum`, set **before** the prefetch, so the driver keeps these pages in GPU memory.
- `cudaMemAdviseSetReadMostly` for `A`, set **after** it is initialized, because the norm kernel only reads `A` (it is a source operand).

In [11]:
%%writefile cuda_advise.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, p;
    parse_args(argc, argv, &blockSize, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    advise_gpu(A, bytes, cudaMemAdviseSetPreferredLocation, device);           // mem advise
    advise_gpu(sum, sizeof(double), cudaMemAdviseSetPreferredLocation, device);
    prefetch_to_gpu(A, bytes, device);                                         // prefetch
    prefetch_to_gpu(sum, sizeof(double), device);
    init_kernel<<<(int)((n + blockSize - 1) / blockSize), blockSize>>>(A, n); // page creation on the GPU
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    advise_gpu(A, bytes, cudaMemAdviseSetReadMostly, device);                  // A is read-only from now on
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 5: CUDA Prefetch + page creation + mem advise",
                   "CUDA Prefetch+Page creation+memadvise", A, n, p, blockSize, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_advise.cu


In [12]:
!nvcc -O2 -arch=sm_75 -o cuda_advise cuda_advise.cu
!for b in 256 512 1024; do ./cuda_advise $b 24; done

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 1.8216 ms | min = 1.8178 ms | max = 1.8479 ms
Run 1 = 1.8479 ms | avg of runs 2-30 = 1.8206 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 512 | blocks = 32768
Kernel time over 30 runs: avg = 2.1324 ms | min = 2.1141 ms | max = 2.1746 ms
Run 1 = 2.1746 ms | avg of runs 2-30 = 2.1310 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.91e-14 -> CORRECT

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 1024 | blocks = 16384
Kernel time over 30 runs: avg = 2.7801 ms | min = 2.7771 ms | max = 2.8062 ms
Run 1 = 2.8062 ms | avg of runs 2-30 = 2.7792 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.91e-14 -> COR

## Results summary (2^24 elements)

Collected from the runs above. Speedup = C average time / version average time. x86-64, XMM and YMM come from our SIMD project (Visual Studio, Windows) and are added to the README separately.

In [13]:
import pandas as pd
cols = ['version','block','power','run1_ms','avg_ms','min_ms','max_ms','Z','Z_ref','correct']
df = pd.read_csv('results.csv', names=cols)
df = df[df['power'] == 24]
c_avg = df[df['version'] == 'C']['avg_ms'].iloc[-1]
df['speedup_vs_C'] = (c_avg / df['avg_ms']).round(2)
pd.set_option('display.width', 200)
print(df[['version','block','run1_ms','avg_ms','min_ms','max_ms','speedup_vs_C','correct']].round(4).to_string(index=False))

                              version block  run1_ms  avg_ms  min_ms  max_ms  speedup_vs_C correct
                                    C     -  28.3190 43.2398 24.6698 73.2367          1.00    PASS
                         CUDA Unified   256  37.4334  3.4607  2.2836 37.4334         12.49    PASS
                         CUDA Unified   512  32.7905  3.2900  2.1829 32.7905         13.14    PASS
                         CUDA Unified  1024  35.2356  3.9207  2.8385 35.2356         11.03    PASS
                        CUDA Prefetch   256   2.6215  2.2967  2.2838  2.6215         18.83    PASS
                        CUDA Prefetch   512   3.1533  2.8120  2.7873  3.1533         15.38    PASS
                        CUDA Prefetch  1024   3.3524  3.0398  3.0250  3.3524         14.22    PASS
          CUDA Prefetch+Page creation   256   2.5518  2.4697  2.4639  2.5518         17.51    PASS
          CUDA Prefetch+Page creation   512   2.8260  2.7943  2.5963  2.8260         15.47    PASS
          

## Profiling — Nsight Systems (nsys), all CUDA versions

Block size 256. `--stats=true` prints the kernel summary, the Unified Memory transfer summary (`cuda_gpu_mem_time_sum` / `cuda_gpu_mem_size_sum`) and the CUDA API summary. Take screenshots of these for the README. The UM page-fault options show where migrations happen (expected: many in Version 2, few or none in Versions 4 and 5).

In [14]:
!nsys profile -o nsys_cuda_unified --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_unified 256 24

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 5.0135 ms | min = 2.7326 ms | max = 62.9736 ms
Run 1 = 62.9736 ms | avg of runs 2-30 = 3.0149 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

Generating '/tmp/nsys-report-f902.qdstrm'
[ 1/11] [========================100%] nsys_cuda_unified.nsys-rep
[ 2/11] [========================100%] nsys_cuda_unified.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_unified.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)        Name     
 --------  ---------------  ---------  ------------  ------------  --------  -----------  ------------  --------------
     91.4    2,028,396,402         97  20,911,303.1  10,083

In [15]:
!nsys profile -o nsys_cuda_prefetch --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_prefetch 256 24

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 3.5553 ms | min = 3.5371 ms | max = 3.9888 ms
Run 1 = 3.9888 ms | avg of runs 2-30 = 3.5403 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

Generating '/tmp/nsys-report-2f61.qdstrm'
[ 1/11] [========================100%] nsys_cuda_prefetch.nsys-rep
[ 2/11] [========================100%] nsys_cuda_prefetch.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_prefetch.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  -----------  ------------  ----------------------
     91.2    1,878,038,711         90  20,867,096.8  1

In [16]:
!nsys profile -o nsys_cuda_pages --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_pages 256 24

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 2.6089 ms | min = 2.6027 ms | max = 2.6377 ms
Run 1 = 2.6377 ms | avg of runs 2-30 = 2.6079 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

Generating '/tmp/nsys-report-dbcd.qdstrm'
[ 1/11] [========================100%] nsys_cuda_pages.nsys-rep
[ 2/11] [========================100%] nsys_cuda_pages.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_pages.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  -----------  ------------  ----------------------
     89.8    1,485,597,993         66  22,509,060.5  10,085,765.5

In [17]:
!nsys profile -o nsys_cuda_advise --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_advise 256 24

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 2.4423 ms | min = 2.4134 ms | max = 2.4942 ms
Run 1 = 2.4942 ms | avg of runs 2-30 = 2.4405 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

Generating '/tmp/nsys-report-0eb4.qdstrm'
[ 1/11] [========================100%] nsys_cuda_advise.nsys-rep
[ 2/11] [========================100%] nsys_cuda_advise.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_advise.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  -----------  ------------  ----------------------
     87.4    1,300,612,413         56  23,225,221.7  10,085,909.5

## Profiling — Nsight Compute (ncu), all CUDA versions

Profiles one `norm_kernel` launch per version (`-k norm_kernel --launch-count 1`) to keep it fast. Note: the kernel times printed by the program during this cell are slowed down by the profiler, so the benchmark numbers come from the cells above, not from here.

In [18]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_unified 256 24

==PROF== Connected to process 1207 (/content/cuda_unified)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 86.1053 ms | min = 2.5800 ms | max = 2506.6877 ms
Run 1 = 2506.6877 ms | avg of runs 2-30 = 2.6370 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

==PROF== Disconnected from process 1207
[1207] cuda_unified@127.0.0.1
  norm_kernel(const double *, long long, double *) (65536, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       584.99
    Elapsed Cycles      

In [19]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_prefetch 256 24

==PROF== Connected to process 1267 (/content/cuda_prefetch)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 76.6068 ms | min = 2.5682 ms | max = 2222.3101 ms
Run 1 = 2222.3101 ms | avg of runs 2-30 = 2.6171 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

==PROF== Disconnected from process 1267
[1267] cuda_prefetch@127.0.0.1
  norm_kernel(const double *, long long, double *) (65536, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                

In [20]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_pages 256 24

==PROF== Connected to process 1324 (/content/cuda_pages)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 80.4461 ms | min = 2.5667 ms | max = 2329.1875 ms
Run 1 = 2329.1875 ms | avg of runs 2-30 = 2.9033 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

==PROF== Disconnected from process 1324
[1324] cuda_pages@127.0.0.1
  norm_kernel(const double *, long long, double *) (65536, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle   

In [21]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_advise 256 24

==PROF== Connected to process 1384 (/content/cuda_advise)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 256 | blocks = 65536
Kernel time over 30 runs: avg = 76.3322 ms | min = 2.4822 ms | max = 2216.0347 ms
Run 1 = 2216.0347 ms | avg of runs 2-30 = 2.5494 ms
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

==PROF== Disconnected from process 1384
[1384] cuda_advise@127.0.0.1
  norm_kernel(const double *, long long, double *) (65536, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle    